In [ ]:
# ============================================================
# QUAD + QUINT FEATURE SEARCH
# ============================================================

import pandas as pd
import numpy as np

from itertools import combinations

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss,
    ConfusionMatrixDisplay
)


# ============================================================
# SETTINGS
# ============================================================

DATA_PATH = "../data/processed/features_v6.csv"

TARGET = "FTR"

SEASON_COL = "Season"

# Seasons used for walk-forward evaluation
TEST_SEASONS = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]

# ------------------------------------------------------------
# Original baseline
# ------------------------------------------------------------

BASELINE_FEATURES = [
    "EloDiff",
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5"
]

# ------------------------------------------------------------
# All available features
# ------------------------------------------------------------

ALL_FEATURES = [
    "EloDiff",
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5",
    "GDPerGameDiff",
    "PPGDiff",
    "GoalsPerGameDiff",
    "GoalsAgainstPerGameDiff",
    "ShotDiffLast5",
    "GoalDiffLast5",
    "HomeElo",
    "AwayElo",
    "HomePPG",
    "AwayPPG",
    "HomeGDPerGame",
    "AwayGDPerGame",
    "HomeGoalsPerGame",
    "AwayGoalsAgainstPerGame",
    "HomePointsLast5",
    "AwayPointsLast5",
    "HomeGoalsLast5",
    "AwayGoalsLast5",
    "HomeGoalsAgainstLast5",
    "AwayGoalsAgainstLast5",
    "HomeShotsAgainstLast5",
    "AwayShotsAgainstLast5",
    "HomeShotsOnTargetLast5",
    "AwayShotsOnTargetLast5",
    "HomeShotsOnTargetAgainstLast5",
    "AwayShotsOnTargetAgainstLast5",
    "XGForDiffPg",
    "XGAgainstDiffPg",
    "XGDDiff",
    "XGForDiffLast5",
    "XGAgainstDiffLast5",
    "XGDDiffLast5"
]


# ============================================================
# PROMISING TRIPLES FROM PREVIOUS SEARCH
# ============================================================

STARTING_TRIPLES = [
    (
        "EloDiff",
        "AwayPointsLast5",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "AwayGoalsLast5",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "AwayElo",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "HomeElo",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "HomeGDPerGame",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "HomeShotsOnTargetLast5",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "ShotOTDiffLast5",
        "HomeShotsAgainstLast5"
    ),
    
    (
        "EloDiff",
        "ShotOTDiffLast5",
        "XGAgainstDiffLast5"
    ),
    
    (
        "EloDiff",
        "ShotDiffLast5",
        "HomeShotsAgainstLast5"
    ),
    
    (
        "EloDiff",
        "GoalDiffLast5",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "HomeShotsOnTargetAgainstLast5",
        "XGDDiffLast5"
    ),
    
    (
        "EloDiff",
        "HomeShotsAgainstLast5",
        "XGDDiffLast5"
    )
]


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("LOADED DATA")
print("=" * 70)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = (
    ALL_FEATURES
    + [TARGET, SEASON_COL]
)

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# CLEAN TARGET
# ============================================================

df = df.dropna(
    subset=[TARGET]
).copy()

print()
print(f"Rows after target cleaning: {len(df)}")


# ============================================================
# CHECK TARGET
# ============================================================

print()
print("Target distribution:")

print(
    df[TARGET]
    .value_counts()
    .sort_index()
)


# ============================================================
# CHECK SEASONS
# ============================================================

print()
print("Games by season:")

print(
    df[SEASON_COL]
    .value_counts()
    .sort_index()
)


# ============================================================
# MODEL
# ============================================================

def create_model():

    return Pipeline([
        
        (
            "scaler",
            StandardScaler()
        ),
        
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
        
    ])


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_feature_set(feature_set):

    feature_set = list(feature_set)

    season_results = []

    all_predictions = []
    all_actual = []

    for test_season in TEST_SEASONS:

        # ----------------------------------------------------
        # Training data
        # ----------------------------------------------------

        train_df = df[
            df[SEASON_COL] < test_season
        ].copy()

        # ----------------------------------------------------
        # Test data
        # ----------------------------------------------------

        test_df = df[
            df[SEASON_COL] == test_season
        ].copy()

        # ----------------------------------------------------
        # Safety checks
        # ----------------------------------------------------

        if len(train_df) == 0:
            continue

        if len(test_df) == 0:
            continue

        # ----------------------------------------------------
        # Remove rows with missing feature values
        # ----------------------------------------------------

        train_mask = (
            train_df[feature_set]
            .notna()
            .all(axis=1)
        )

        test_mask = (
            test_df[feature_set]
            .notna()
            .all(axis=1)
        )

        train_df = train_df.loc[train_mask]
        test_df = test_df.loc[test_mask]

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if len(train_df) == 0:
            raise ValueError(
                f"No training rows for season "
                f"{test_season} using features:\n"
                f"{feature_set}"
            )

        if len(test_df) == 0:
            raise ValueError(
                f"No test rows for season "
                f"{test_season} using features:\n"
                f"{feature_set}"
            )

        # ----------------------------------------------------
        # X / y
        # ----------------------------------------------------

        X_train = train_df[feature_set]
        y_train = train_df[TARGET]

        X_test = test_df[feature_set]
        y_test = test_df[TARGET]

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        model = create_model()

        model.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions = model.predict(X_test)

        probabilities = model.predict_proba(X_test)

        classes = model.classes_

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        logloss = log_loss(
            y_test,
            probabilities,
            labels=classes
        )

        # ----------------------------------------------------
        # Brier score
        #
        # Multiclass Brier:
        # sum of squared probability errors
        # divided by number of observations
        # ----------------------------------------------------

        y_test_array = y_test.to_numpy()

        class_to_index = {
            cls: i
            for i, cls in enumerate(classes)
        }

        y_encoded = np.array([
            class_to_index[value]
            for value in y_test_array
        ])

        one_hot = np.zeros(
            (
                len(y_encoded),
                len(classes)
            )
        )

        one_hot[
            np.arange(len(y_encoded)),
            y_encoded
        ] = 1

        brier = np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot
                ) ** 2,
                axis=1
            )
        )

        # ----------------------------------------------------
        # Store season result
        # ----------------------------------------------------

        season_results.append({
            "Season": test_season,
            "Games": len(test_df),
            "Accuracy": accuracy,
            "LogLoss": logloss,
            "Brier": brier
        })

        # ----------------------------------------------------
        # Store predictions
        # ----------------------------------------------------

        all_predictions.extend(
            probabilities
        )

        all_actual.extend(
            y_encoded
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    season_results_df = pd.DataFrame(
        season_results
    )

    if len(season_results_df) == 0:
        raise ValueError(
            "No season results were generated."
        )

    mean_accuracy = (
        season_results_df["Accuracy"]
        .mean()
    )

    weighted_accuracy = (
        np.average(
            season_results_df["Accuracy"],
            weights=season_results_df["Games"]
        )
    )

    mean_logloss = (
        season_results_df["LogLoss"]
        .mean()
    )

    mean_brier = (
        season_results_df["Brier"]
        .mean()
    )

    return {
        "MeanAccuracy": mean_accuracy,
        "WeightedAccuracy": weighted_accuracy,
        "MeanLogLoss": mean_logloss,
        "MeanBrier": mean_brier,
        "SeasonResults": season_results_df
    }


# ============================================================
# BASELINE
# ============================================================

print()
print("=" * 70)
print("BASELINE")
print("=" * 70)

print()
print("Baseline features:")

for feature in BASELINE_FEATURES:
    print(f"- {feature}")

baseline_result = evaluate_feature_set(
    BASELINE_FEATURES
)

print()
print(
    f"Mean Accuracy: "
    f"{baseline_result['MeanAccuracy']:.6f}"
)

print(
    f"Weighted Accuracy: "
    f"{baseline_result['WeightedAccuracy']:.6f}"
)

print(
    f"Mean Log Loss: "
    f"{baseline_result['MeanLogLoss']:.6f}"
)

print(
    f"Mean Brier: "
    f"{baseline_result['MeanBrier']:.6f}"
)

print()
print("Season results:")

print(
    baseline_result["SeasonResults"]
    .to_string(index=False)
)


# ============================================================
# FUNCTION TO TEST ADDITIONS TO A SET
# ============================================================

def test_additions(
    starting_features,
    candidate_features,
    stage_name
):

    results = []

    starting_features = tuple(
        starting_features
    )

    candidate_features = [
        feature
        for feature in candidate_features
        if feature not in starting_features
    ]

    print()
    print("=" * 70)
    print(stage_name)
    print("=" * 70)

    print(
        f"Starting features: "
        f"{' + '.join(starting_features)}"
    )

    print(
        f"Testing {len(candidate_features)} possible additions..."
    )

    for i, feature in enumerate(
        candidate_features,
        start=1
    ):

        feature_set = (
            starting_features
            + (feature,)
        )

        print(
            f"[{i}/{len(candidate_features)}] "
            f"{feature}"
        )

        metrics = evaluate_feature_set(
            feature_set
        )

        results.append({
            "Features": " + ".join(
                feature_set
            ),
            "FeatureTuple": feature_set,
            "NumFeatures": len(feature_set),
            "AddedFeature": feature,
            "MeanAccuracy": metrics["MeanAccuracy"],
            "WeightedAccuracy": metrics["WeightedAccuracy"],
            "MeanLogLoss": metrics["MeanLogLoss"],
            "MeanBrier": metrics["MeanBrier"]
        })

    results_df = pd.DataFrame(
        results
    )

    results_df = results_df.sort_values(
        by="MeanLogLoss",
        ascending=True
    ).reset_index(drop=True)

    return results_df


# ============================================================
# TEST QUADS
# ============================================================

quad_results_all = []

print()
print("=" * 70)
print("SEARCHING 4-FEATURE COMBINATIONS")
print("=" * 70)

for triple_number, triple in enumerate(
    STARTING_TRIPLES,
    start=1
):

    print()
    print(
        f"TRIPLE {triple_number}/"
        f"{len(STARTING_TRIPLES)}"
    )

    print(
        " + ".join(triple)
    )

    remaining_features = [
        feature
        for feature in ALL_FEATURES
        if feature not in triple
    ]

    triple_results = test_additions(
        triple,
        remaining_features,
        f"EXPANDING TRIPLE {triple_number}"
    )

    quad_results_all.append(
        triple_results
    )


# ============================================================
# COMBINE QUAD RESULTS
# ============================================================

quad_results = pd.concat(
    quad_results_all,
    ignore_index=True
)

quad_results = (
    quad_results
    .sort_values(
        by="MeanLogLoss",
        ascending=True
    )
    .drop_duplicates(
        subset=["Features"]
    )
    .reset_index(drop=True)
)


# ============================================================
# DISPLAY BEST QUADS
# ============================================================

print()
print("=" * 70)
print("BEST 4-FEATURE COMBINATIONS")
print("=" * 70)

print()

display_columns = [
    "Features",
    "NumFeatures",
    "MeanAccuracy",
    "MeanLogLoss",
    "MeanBrier"
]

print(
    quad_results[
        display_columns
    ]
    .head(25)
    .to_string(index=False)
)


# ============================================================
# HOW MANY QUADS TO EXPAND?
# ============================================================

# Expand the best N quads into 5-feature combinations.
#
# 20 is enough to give us a good search without exploding
# the number of Logistic Regression fits.

TOP_QUADS_TO_EXPAND = 20

top_quads = quad_results.head(
    TOP_QUADS_TO_EXPAND
).copy()


# ============================================================
# TEST QUINTS
# ============================================================

quint_results_all = []

print()
print("=" * 70)
print("SEARCHING 5-FEATURE COMBINATIONS")
print("=" * 70)

print(
    f"Expanding top "
    f"{TOP_QUADS_TO_EXPAND} quads."
)


for quad_number, row in enumerate(
    top_quads.iterrows(),
    start=1
):

    _, quad_row = row

    quad = tuple(
        quad_row["FeatureTuple"]
    )

    print()
    print(
        f"QUAD {quad_number}/"
        f"{len(top_quads)}"
    )

    print(
        " + ".join(quad)
    )

    remaining_features = [
        feature
        for feature in ALL_FEATURES
        if feature not in quad
    ]

    quint_results = test_additions(
        quad,
        remaining_features,
        f"EXPANDING QUAD {quad_number}"
    )

    quint_results_all.append(
        quint_results
    )


# ============================================================
# COMBINE QUINT RESULTS
# ============================================================

quint_results = pd.concat(
    quint_results_all,
    ignore_index=True
)

quint_results = (
    quint_results
    .sort_values(
        by="MeanLogLoss",
        ascending=True
    )
    .drop_duplicates(
        subset=["Features"]
    )
    .reset_index(drop=True)
)


# ============================================================
# BEST QUINTS
# ============================================================

print()
print("=" * 70)
print("BEST 5-FEATURE COMBINATIONS")
print("=" * 70)

print()

print(
    quint_results[
        display_columns
    ]
    .head(25)
    .to_string(index=False)
)


# ============================================================
# COMBINE QUADS + QUINTS
# ============================================================

all_results = pd.concat(
    [
        quad_results[
            display_columns
        ],
        quint_results[
            display_columns
        ]
    ],
    ignore_index=True
)


# ============================================================
# ADD BASELINE
# ============================================================

baseline_row = pd.DataFrame([{
    "Features": " + ".join(
        BASELINE_FEATURES
    ),
    "NumFeatures": len(
        BASELINE_FEATURES
    ),
    "MeanAccuracy": baseline_result[
        "MeanAccuracy"
    ],
    "MeanLogLoss": baseline_result[
        "MeanLogLoss"
    ],
    "MeanBrier": baseline_result[
        "MeanBrier"
    ]
}])


comparison_results = pd.concat(
    [
        baseline_row,
        all_results
    ],
    ignore_index=True
)


# ============================================================
# SORT EVERYTHING BY LOG LOSS
# ============================================================

comparison_results = (
    comparison_results
    .sort_values(
        by="MeanLogLoss",
        ascending=True
    )
    .reset_index(drop=True)
)


# ============================================================
# BEST OVERALL
# ============================================================

best_row = (
    comparison_results
    .iloc[0]
)

best_features_string = (
    best_row["Features"]
)

best_features = [
    feature.strip()
    for feature in
    best_features_string.split("+")
]


# ============================================================
# METRIC CHANGES VS BASELINE
# ============================================================

accuracy_change = (
    best_row["MeanAccuracy"]
    - baseline_result["MeanAccuracy"]
)

logloss_change = (
    best_row["MeanLogLoss"]
    - baseline_result["MeanLogLoss"]
)

brier_change = (
    best_row["MeanBrier"]
    - baseline_result["MeanBrier"]
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()
print("=" * 70)
print("BEST OVERALL RESULT")
print("=" * 70)

print()

print(
    f"Features: "
    f"{best_features_string}"
)

print(
    f"Number of features: "
    f"{int(best_row['NumFeatures'])}"
)

print()

print(
    f"Mean Accuracy: "
    f"{best_row['MeanAccuracy']:.6f}"
)

print(
    f"Mean Log Loss: "
    f"{best_row['MeanLogLoss']:.6f}"
)

print(
    f"Mean Brier: "
    f"{best_row['MeanBrier']:.6f}"
)

print()

print("Compared with baseline:")

print(
    f"Accuracy change: "
    f"{accuracy_change:+.6f}"
)

print(
    f"Log Loss change: "
    f"{logloss_change:+.6f}"
)

print(
    f"Brier change: "
    f"{brier_change:+.6f}"
)


# ============================================================
# FINAL SEASON EVALUATION
# ============================================================

print()
print("=" * 70)
print("BEST MODEL SEASON RESULTS")
print("=" * 70)

best_metrics = evaluate_feature_set(
    best_features
)

print()

print(
    best_metrics["SeasonResults"]
    .to_string(index=False)
)


# ============================================================
# SAVE QUAD RESULTS
# ============================================================

quad_save = quad_results.copy()

if "FeatureTuple" in quad_save.columns:
    quad_save = quad_save.drop(
        columns=["FeatureTuple"]
    )

quad_path = (
    "../data/processed/"
    "quad_feature_search_v6.csv"
)

quad_save.to_csv(
    quad_path,
    index=False
)


# ============================================================
# SAVE QUINT RESULTS
# ============================================================

quint_save = quint_results.copy()

if "FeatureTuple" in quint_save.columns:
    quint_save = quint_save.drop(
        columns=["FeatureTuple"]
    )

quint_path = (
    "../data/processed/"
    "quint_feature_search_v6.csv"
)

quint_save.to_csv(
    quint_path,
    index=False
)


# ============================================================
# SAVE COMBINED RESULTS
# ============================================================

combined_path = (
    "../data/processed/"
    "quad_quint_feature_search_v6.csv"
)

comparison_results.to_csv(
    combined_path,
    index=False
)


# ============================================================
# SAVE FINAL FEATURE LIST
# ============================================================

final_features_path = (
    "../data/processed/"
    "final_features_v6.txt"
)

with open(
    final_features_path,
    "w"
) as f:

    for feature in best_features:
        f.write(
            feature + "\n"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print()
print(
    f"Quads: {quad_path}"
)

print(
    f"Quints: {quint_path}"
)

print(
    f"Combined: {combined_path}"
)

print(
    f"Final features: {final_features_path}"
)

print()
print("=" * 70)
print("FINAL FEATURE LIST")
print("=" * 70)

for i, feature in enumerate(
    best_features,
    start=1
):

    print(
        f"{i}. {feature}"
    )

print()
print("=" * 70)

LOADED DATA
Loaded: ../data/processed/features_v6.csv
Rows: 1900
Columns: 236

Rows after target cleaning: 1900

Target distribution:
FTR
A    607
D    454
H    839
Name: count, dtype: int64

Games by season:
Season
21-22    380
22-23    380
23-24    380
24-25    380
25-26    380
Name: count, dtype: int64

BASELINE

Baseline features:
- EloDiff
- ShotOTDiffLast5
- GoalAgainstDiffLast5

Mean Accuracy: 0.541447
Weighted Accuracy: 0.541447
Mean Log Loss: 0.990174
Mean Brier: 0.590873

Season results:
Season  Games  Accuracy  LogLoss    Brier
 22-23    380  0.557895 0.996115 0.593924
 23-24    380  0.571053 0.932963 0.548853
 24-25    380  0.544737 0.996525 0.596465
 25-26    380  0.492105 1.035092 0.624249

SEARCHING 4-FEATURE COMBINATIONS

TRIPLE 1/12
EloDiff + AwayPointsLast5 + XGDDiffLast5

EXPANDING TRIPLE 1
Starting features: EloDiff + AwayPointsLast5 + XGDDiffLast5
Testing 32 possible additions...
[1/32] ShotOTDiffLast5
[2/32] GoalAgainstDiffLast5
[3/32] GDPerGameDiff
[4/32] PPGDiff

KeyboardInterrupt: 

In [3]:
# ============================================================
# FINAL MODEL EVALUATION - V6
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = "../data/processed/features_v6.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# Make sure dates are correctly parsed
df["MatchDateTime"] = pd.to_datetime(df["MatchDateTime"])

# Sort chronologically
df = df.sort_values("MatchDateTime").reset_index(drop=True)

# Remove rows without target
df = df.dropna(subset=["FTR"]).copy()

print(f"\nRows after target cleaning: {len(df)}")

print("\nTarget distribution:")
print(df["FTR"].value_counts())

print("\nGames by season:")
print(df["Season"].value_counts().sort_index())


# ============================================================
# 2. FINAL FEATURES
# ============================================================

FEATURES = [
    "EloDiff",
    "AwayElo",
    "XGDDiffLast5",
    "AwayGDPerGame",
    "AwayPointsLast5"
]

TARGET = "FTR"

print("\n" + "=" * 70)
print("FINAL FEATURE SET")
print("=" * 70)

for i, feature in enumerate(FEATURES, 1):
    print(f"{i}. {feature}")

print(f"\nNumber of features: {len(FEATURES)}")


# ============================================================
# 3. CHECK FEATURES
# ============================================================

missing_features = [
    feature for feature in FEATURES
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing features from dataset: {missing_features}"
    )

print("\nAll features found.")


# ============================================================
# 4. REMOVE ROWS WITH MISSING FEATURE VALUES
# ============================================================

before = len(df)

df_model = df.dropna(
    subset=FEATURES + [TARGET]
).copy()

after = len(df_model)

print(f"\nRows removed due to missing features: {before - after}")
print(f"Rows available for modelling: {after}")


# ============================================================
# 5. DEFINE MODEL
# ============================================================

def create_model():

    return RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight=None,
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# 6. WALK-FORWARD / SEASON-BY-SEASON EVALUATION
# ============================================================

seasons = sorted(df_model["Season"].unique())

print("\n" + "=" * 70)
print("WALK-FORWARD EVALUATION")
print("=" * 70)

all_predictions = []
season_results = []

# We cannot evaluate the first season because there is
# no previous season to train on.

for season in seasons[1:]:

    train = df_model[
        df_model["Season"] < season
    ].copy()

    test = df_model[
        df_model["Season"] == season
    ].copy()

    X_train = train[FEATURES]
    y_train = train[TARGET]

    X_test = test[FEATURES]
    y_test = test[TARGET]

    print(f"\nSeason: {season}")
    print(f"Training games: {len(train)}")
    print(f"Testing games:  {len(test)}")

    # Create fresh model
    model = create_model()

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities
    y_prob = model.predict_proba(X_test)

    # Ensure probability columns match classes
    classes = model.classes_

    # Accuracy
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    # Log loss
    logloss = log_loss(
        y_test,
        y_prob,
        labels=classes
    )

    # Multiclass Brier score
    brier = np.mean([
        brier_score_loss(
            (y_test == cls).astype(int),
            y_prob[:, i]
        )
        for i, cls in enumerate(classes)
    ])

    print(f"Accuracy: {accuracy:.6f}")
    print(f"Log Loss: {logloss:.6f}")
    print(f"Brier:    {brier:.6f}")

    season_results.append({
        "Season": season,
        "Games": len(test),
        "Accuracy": accuracy,
        "LogLoss": logloss,
        "Brier": brier
    })

    # Store predictions
    prediction_df = test[
        [
            "MatchDateTime",
            "Season",
            "HomeTeam",
            "AwayTeam",
            "FTR"
        ]
    ].copy()

    prediction_df["Prediction"] = y_pred

    for i, cls in enumerate(classes):
        prediction_df[f"Prob_{cls}"] = y_prob[:, i]

    all_predictions.append(prediction_df)


# ============================================================
# 7. COMBINE RESULTS
# ============================================================

season_results_df = pd.DataFrame(
    season_results
)

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)


# ============================================================
# 8. OVERALL METRICS
# ============================================================

y_true = predictions_df["FTR"]
y_pred = predictions_df["Prediction"]

prob_columns = [
    col for col in predictions_df.columns
    if col.startswith("Prob_")
]

y_prob = predictions_df[prob_columns].values

classes = [
    col.replace("Prob_", "")
    for col in prob_columns
]

overall_accuracy = accuracy_score(
    y_true,
    y_pred
)

overall_logloss = log_loss(
    y_true,
    y_prob,
    labels=classes
)

overall_brier = np.mean([
    brier_score_loss(
        (y_true == cls).astype(int),
        y_prob[:, i]
    )
    for i, cls in enumerate(classes)
])


# ============================================================
# 9. PRINT RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL MODEL RESULTS")
print("=" * 70)

print("\nFeatures:")

for feature in FEATURES:
    print(f"- {feature}")

print("\nOverall:")

print(f"Accuracy: {overall_accuracy:.6f}")
print(f"Log Loss: {overall_logloss:.6f}")
print(f"Brier:    {overall_brier:.6f}")

print("\nSeason results:")

print(
    season_results_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.6f}".format,
            "LogLoss": "{:.6f}".format,
            "Brier": "{:.6f}".format
        }
    )
)


# ============================================================
# 10. CONFUSION MATRIX
# ============================================================

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["H", "D", "A"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual H", "Actual D", "Actual A"],
    columns=["Pred H", "Pred D", "Pred A"]
)

print(cm_df)


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        labels=["H", "D", "A"],
        digits=4
    )
)


# ============================================================
# 12. PREDICTION DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

print("\nActual:")
print(
    y_true.value_counts(normalize=True)
    .sort_index()
)

print("\nPredicted:")
print(
    y_pred.value_counts(normalize=True)
    .sort_index()
)


# ============================================================
# 13. SAVE PREDICTIONS
# ============================================================

PREDICTION_PATH = (
    "../data/processed/"
    "final_model_predictions_v6.csv"
)

predictions_df.to_csv(
    PREDICTION_PATH,
    index=False
)

print(
    f"\nPredictions saved to: {PREDICTION_PATH}"
)


# ============================================================
# 14. SAVE FINAL MODEL FEATURES
# ============================================================

FEATURE_PATH = (
    "../data/processed/"
    "final_features_v6.txt"
)

with open(FEATURE_PATH, "w") as f:

    for i, feature in enumerate(FEATURES, 1):
        f.write(f"{i}. {feature}\n")

print(
    f"Features saved to: {FEATURE_PATH}"
)


# ============================================================
# 15. MOST CONFIDENT WRONG PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("MOST CONFIDENT WRONG PREDICTIONS")
print("=" * 70)

# Probability assigned to the model's prediction
predicted_prob = []

for _, row in predictions_df.iterrows():

    pred = row["Prediction"]

    predicted_prob.append(
        row[f"Prob_{pred}"]
    )

predictions_df["PredictionProbability"] = predicted_prob

wrong = predictions_df[
    predictions_df["FTR"] != predictions_df["Prediction"]
].copy()

wrong = wrong.sort_values(
    "PredictionProbability",
    ascending=False
)

columns_to_show = [
    "MatchDateTime",
    "Season",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "Prediction",
    "PredictionProbability",
    "Prob_H",
    "Prob_D",
    "Prob_A"
]

print(
    wrong[columns_to_show]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 16. MOST UNCERTAIN PREDICTIONS
# ============================================================

print("\n" + "=" * 70)
print("MOST UNCERTAIN PREDICTIONS")
print("=" * 70)

prob_values = predictions_df[
    ["Prob_H", "Prob_D", "Prob_A"]
].values

predictions_df["MaxProbability"] = (
    prob_values.max(axis=1)
)

uncertain = predictions_df.sort_values(
    "MaxProbability"
)

print(
    uncertain[
        [
            "MatchDateTime",
            "Season",
            "HomeTeam",
            "AwayTeam",
            "FTR",
            "Prediction",
            "Prob_H",
            "Prob_D",
            "Prob_A",
            "MaxProbability"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 17. ACCURACY BY ACTUAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("ACCURACY BY RESULT")
print("=" * 70)

for result in ["H", "D", "A"]:

    subset = predictions_df[
        predictions_df["FTR"] == result
    ]

    accuracy = (
        subset["Prediction"] == result
    ).mean()

    print(
        f"{result}: "
        f"{accuracy:.6f} "
        f"({len(subset)} games)"
    )


# ============================================================
# 18. ACCURACY BY SEASON AND RESULT
# ============================================================

print("\n" + "=" * 70)
print("ACCURACY BY SEASON / RESULT")
print("=" * 70)

season_result_rows = []

for season in sorted(
    predictions_df["Season"].unique()
):

    season_data = predictions_df[
        predictions_df["Season"] == season
    ]

    for result in ["H", "D", "A"]:

        subset = season_data[
            season_data["FTR"] == result
        ]

        if len(subset) > 0:

            accuracy = (
                subset["Prediction"] == result
            ).mean()

        else:

            accuracy = np.nan

        season_result_rows.append({
            "Season": season,
            "Result": result,
            "Games": len(subset),
            "Accuracy": accuracy
        })

season_result_df = pd.DataFrame(
    season_result_rows
)

print(
    season_result_df.to_string(
        index=False,
        formatters={
            "Accuracy": lambda x:
                f"{x:.6f}" if pd.notna(x) else "N/A"
        }
    )
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"""
Final feature set:

1. EloDiff
2. AwayElo
3. XGDDiffLast5
4. AwayGDPerGame
5. AwayPointsLast5

Overall Accuracy : {overall_accuracy:.6f}
Overall Log Loss : {overall_logloss:.6f}
Overall Brier    : {overall_brier:.6f}

Baseline:

Accuracy         : 0.541447
Log Loss         : 0.990174
Brier            : 0.590873

Final model:

Accuracy change  : {overall_accuracy - 0.541447:+.6f}
Log Loss change  : {overall_logloss - 0.990174:+.6f}
Brier change     : {overall_brier - 0.590873:+.6f}
""")

print("=" * 70)
print("DONE")
print("=" * 70)

Loaded: ../data/processed/features_v6.csv
Rows: 1900
Columns: 236

Rows after target cleaning: 1900

Target distribution:
FTR
H    839
A    607
D    454
Name: count, dtype: int64

Games by season:
Season
21-22    380
22-23    380
23-24    380
24-25    380
25-26    380
Name: count, dtype: int64

FINAL FEATURE SET
1. EloDiff
2. AwayElo
3. XGDDiffLast5
4. AwayGDPerGame
5. AwayPointsLast5

Number of features: 5

All features found.

Rows removed due to missing features: 0
Rows available for modelling: 1900

WALK-FORWARD EVALUATION

Season: 22-23
Training games: 380
Testing games:  380
Accuracy: 0.510526
Log Loss: 1.047007
Brier:    0.207714

Season: 23-24
Training games: 760
Testing games:  380
Accuracy: 0.557895
Log Loss: 0.971427
Brier:    0.191654

Season: 24-25
Training games: 1140
Testing games:  380
Accuracy: 0.531579
Log Loss: 0.995922
Brier:    0.198691

Season: 25-26
Training games: 1520
Testing games:  380
Accuracy: 0.471053
Log Loss: 1.067869
Brier:    0.213482

FINAL MODEL RESU

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds)